In [29]:
import sys
sys.path.append('/home/jovyan/BenchmarkingML4KGE_extraction')
import os
from pathlib import Path

from utils.XMLParser import XMLParser
from utils import llm_preprocessing
from utils.experimentation_utils import get_process_info, get_gpu_usage, calcular_bertscore_listas
import json
import requests
import torch
from tqdm.auto import tqdm

import time
from bert_score import score
import logging
from transformers import logging as transformers_logging
transformers_logging.set_verbosity_error()


from gliner import GLiNER
from utils.grobid_service import GrobidService

import ollama
from ollama import Client, ResponseError
from transformers import AutoModelForCausalLM, AutoTokenizer
import accelerate

with open('../data/dataset_con_rutas_xml.json', 'r', encoding='utf-8') as f:
    kge_dataset=json.load(f)

service=GrobidService()

In [30]:
model_name = "Qwen/Qwen3-1.7B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
max_context_tokens = 32768 - 2048


def extract_with_qwen(text, question):
    chat = [
        {"role": "system", "content":
            "You are an assistant for QA tasks. Use only provided context."
        },
        {"role": "user", "content": f"Context chunk: {text}"}
    ]
    
    prompt=(f"Given the following question: {question}"+"Return the answer only in a Python list format, i.e. ['A','B']. You must return an empty list if there is no model presented")
    chat.append({"role":"user","content":prompt})
    predictions=llm_preprocessing.query_model_return_list(model,chat,tokenizer,local=True)

    return predictions

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

In [31]:
model_name="llama3"
client=Client(timeout=600.0)
def extract_with_llama(text, question):
    chat = [
        {"role": "system", "content":
            "You are an assistant for QA tasks. Use only provided context."
        },
        {"role": "user", "content": f"Context chunk: {text}"}
    ]

    prompt=(f"Given the following question: {question}"+"Return the answer only in a Python list format, i.e. ['A','B']. You must return an empty list if there is no model presented")
    chat.append({"role":"user","content":prompt})
    predictions = "[]"
    
    try:
        response=client.chat(model=model_name, messages=chat)
        predictions=response['message']['content']
    except Exception as e:
        print(f"Timeout error llama")
        predictions = []

    return predictions

In [32]:
def extract_taxonomy_with_llama(text):
    model_context = '''The semantic matching models and The translation models only use the structure information of internal facts in KGs. The semantic matching models generally use semantic matching-based scoring functions and further consists of tensor/matrix factorization models and neural network models. The translation models apply distance-based scoring functions.
    While Internal side information inside KGs and External extra information outside KGs outside KGs cooperate with additional information (the inside or outside information of KGs except for the structure information) to achieve KGC. Internal side information inside KGs involved in KGs, including node attributes information, entity-related information, relation-related information, neighborhood information, relational path information; External extra information outside KGs outside KGs, mainly including two aspects: rule-based KGC and third-party data sources-based KGC. 
    And if it is not any of the previous models, then it is Other KGC technologies.'''
    question = "Given the model definitions mentioned before, choose one of the following taxonomy as the taxonomy of the model mentioned in this paper: Semantic matching model, Translation models, Internal side information inside KGs model, External extra information outside KGs or Other KGC Technologies ? "

    chat = [
        {"role": "system", "content":
            "You are an assistant for QA tasks. Use only provided context."
        },
        {"role": "user", "content": f"Context chunk: {text}"}
    ]

    prompt = (
           f"Now, given this context for model taxonomy: {model_context} Answer this question: {question} "
           +"Give back the answer only and only in a correct Python list format, for example: ['A']. If you don't know the answer, just return an empty list."

        )
    chat.append({"role":"user","content":prompt})
    predictions = "[]"
    try:
        response=client.chat(model=model_name, messages=chat)
        predictions=response['message']['content']
    except Exception as e:
        print(f"Timeout error llama tax")
        

    return predictions

In [33]:
# dataset_reducido = kge_dataset[:5]

In [ ]:
paper_evaluation_data = []
tiempos = []

dataset_ext_scores = []
task_ext_scores = []
model_ext_scores = []
implementation_ext_scores = []
taxonomy_ext_scores = []


base_path = Path(os.getcwd()).parent

for paper in tqdm(kge_dataset, desc="Processing"):
    xml_path = paper.get('xml_file')

    # Extraer el groundtruth de cada campo para este paper
    task_gt = paper.get('tasks',[])
    dataset_gt = paper.get('Datasets',[])
    model_gt = []
    method_list = paper["methods"]
    for method in method_list:
        method_name = method["name"]
        model_gt.append(method_name)
    implementation_gt = paper.get('repo_url','')
    category_gt=paper.get('category',[])
    
    if not xml_path:
        continue
    archive=Path(xml_path)
    filename=archive.name
    filename=filename.replace("\\","/")
    base_path = Path(os.getcwd()).parent
    xml_path = base_path / "data" / filename

    parser=XMLParser(xml_path)
    abstract=parser.get_abstract()
    full_text=parser.get_full_text()

    if not abstract or not full_text:
        continue
        print('An error occurred while processing')

    #init_mem_usage, _ = get_process_info()
    #init_vram_usage = get_gpu_usage()
    init_time=time.time()
    
    # Métodos precisos
    dataset_question = "What are the datasets used in this paper?"
    extracted_datasets = extract_with_qwen(full_text, dataset_question)
    task_question = "What are the tasks addressed in this paper?"
    extracted_tasks = extract_with_qwen(abstract, task_question)
    model_question = "What is the name of the model presented in this paper?"
    extracted_models = extract_with_llama(abstract, model_question)
    implementation_question = "Is there a URL in the paper providing the implementation of the model?"
    extracted_implementations = extract_with_llama(full_text, implementation_question)
    extracted_category = extract_taxonomy_with_llama(full_text)
    print(extracted_category)

    #end_mem_usage, cpu_usage = get_process_info()
    #end_vram_usage = get_gpu_usage()
    mem_usage, cpu_usage = get_process_info()
    vram_usage = get_gpu_usage()
    end_time = time.time()

    #total_mem_usage = end_mem_usage - init_mem_usage
    #total_vram_usage = end_vram_usage - init_vram_usage
    total_time = end_time - init_time
    tiempos.append(total_time)

    dataset_f1 = calcular_bertscore_listas(extracted_datasets,dataset_gt)
    dataset_ext_scores.append(dataset_f1)
    task_f1 = calcular_bertscore_listas(extracted_tasks,task_gt)
    task_ext_scores.append(task_f1)
    model_f1 = calcular_bertscore_listas(extracted_models,model_gt)
    model_ext_scores.append(model_f1)
    implementation_f1 = calcular_bertscore_listas(extracted_implementations,implementation_gt)
    implementation_ext_scores.append(implementation_f1)
    taxonomy_f1 = calcular_bertscore_listas(extracted_category,category_gt)
    taxonomy_ext_scores.append(taxonomy_f1)

    average_f1 = (dataset_f1 + task_f1 + model_f1 + implementation_f1 + taxonomy_f1)/5

    paper_evaluation_data.append({
            "title": paper.get('title'),
            "time": total_time,
            "vram_usage": vram_usage,
            "ram_mem_usage": mem_usage,
            "cpu_usage": cpu_usage,
            "dataset_f1": dataset_f1,
            "task_f1": task_f1,
            "model_f1": model_f1,
            "implementation_f1": implementation_f1,
            "taxonomy_f1": taxonomy_f1,
            "average_f1": average_f1,
    })

average_time = sum(tiempos) / len(tiempos) if tiempos else 0

evaluation_data = {
    "total_time": sum(tiempos),
    "average_time": average_time,
    "paper_evaluation_data": paper_evaluation_data
}

with open("accurate_extraction_results.json", "w", encoding="utf-8") as f:
		json.dump(evaluation_data, f, indent=4)




NameError: name 'resultados' is not defined